# **Laboratorio 1**

Laura Sanchez Bernal - 202411353

Baruc ....

## **Actividades a realizar**

2. Justificando las decisiones tomadas con base en los resultados obtenidos en el paso anterior y de acuerdo con el modelo que van a construir.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

from sklearn.impute import SimpleImputer

datos = pd.read_csv('./data/Datos lab 1.csv', )
datosPrueba = pd.read_csv('./data/Datos Test Lab 1.csv', )

data = datos.copy()
dataPrueba = datosPrueba.copy()

In [16]:
from importlib.metadata import version
print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Matplotlib: {version('matplotlib')}")
print(f"Versión de Seaborn: {version('seaborn')}")
print(f"Versión de Scikit learn: {version('scikit-learn')}")

Versión de Pandas: 3.0.3
Versión de Matplotlib: 3.11.0
Versión de Seaborn: 0.13.2
Versión de Scikit learn: 1.9.0


## **Unicidad**

1. **Filas duplicadas:** 4 filas idénticas en las 27 columnas, es un caso sin ambigüedad. Se conserva solo la primera con drop_duplicates()


In [17]:
n_duplicados = int(data.duplicated(keep=False).sum())
print(f"Número de registros duplicados: {n_duplicados}")
data = data.drop_duplicates(keep='first')


Número de registros duplicados: 8


## **Consistencia**

1. **Sector_viento, mes y estacion_anio:** Las columnas presentaban muchas variantes distintas para las mismas categorías como mezcla de mayúsculas/minúsculas, español/inglés, abreviado/completo. Por ejemplo, 27 categorías para sector_viento cuando solo deberían existir 8. Aquí el problema es complejo puesto que hay traducción de idioma y formato abreviado, así que se hizo un diccionario de mapeo ya que es la estrategia que el notebook recomienda para "múltiples variantes a unificar". Se usó .replace() en lugar de .map() porque .replace() conserva sin cambios cualquier valor que no esté explícitamente en el diccionario, mientras que .map() lo convertiría en NaN. Esto es más seguro si existiera alguna variante que no detectamos en la exploración inicial, evitando pérdida silenciosa de información.

In [45]:
data['sector_viento'] = data['sector_viento'].str.strip().str.upper()

mapeo_sector_viento_v2 = {
    'NORTH': 'N',
    'EAST': 'E',
    'SURESTE': 'SE',
    'OESTE': 'O',
    'NOROESTE': 'NO',
}
data['sector_viento'] = data['sector_viento'].replace(mapeo_sector_viento_v2)

print(data['sector_viento'].value_counts())
print(f"\nTotal categorías: {data['sector_viento'].nunique()}")

sector_viento
SO    800
S     541
NE    447
O     307
N     132
E      78
SE     68
NO     45
Name: count, dtype: int64

Total categorías: 8


In [46]:
data['mes'] = data['mes'].str.strip().str.lower()

mapeo_mes_v2 = {
    'january': 'enero', 'jan': 'enero',
    'february': 'febrero', 'feb': 'febrero',
    'march': 'marzo', 'mar': 'marzo',
    'april': 'abril', 'apr': 'abril',
    'may': 'mayo',
    'june': 'junio', 'jun': 'junio',
    'july': 'julio', 'jul': 'julio',
    'august': 'agosto', 'aug': 'agosto',
    'september': 'septiembre', 'sep': 'septiembre', 'sept': 'septiembre',
    'october': 'octubre', 'oct': 'octubre',
    'november': 'noviembre', 'nov': 'noviembre',
    'december': 'diciembre', 'dec': 'diciembre',
}
data['mes'] = data['mes'].replace(mapeo_mes_v2)

print(data['mes'].value_counts())
print(f"\nTotal categorías: {data['mes'].nunique()}")

mes
enero         301
septiembre    221
diciembre     202
noviembre     200
marzo         200
julio         199
agosto        197
mayo          186
octubre       183
febrero       182
junio         178
abril         169
Name: count, dtype: int64

Total categorías: 12


In [54]:
data['estacion_anio'] = data['estacion_anio'].str.strip().str.lower()

mapeo_estacion_anio = {
    'primaveraa': 'primavera', 'primav' : 'primavera', 'spring' : 'primavera',
    'invernio': 'invierno', 'winter': 'invierno',
    'otono': 'otoño', 'autumn': 'otoño', 'fall': 'otoño',
    'berano': 'verano', 'verno': 'verano', 'summer': 'verano',
}
data['estacion_anio'] = data['estacion_anio'].replace(mapeo_estacion_anio)

valores_invalidos = ['inverano', 'estacion_desconocida', 'verano_invierno', 'east']
n_invalidos = data['estacion_anio'].isin(valores_invalidos).sum()
print(f"Valores inválidos convertidos a NaN: {n_invalidos}")
data.loc[data['estacion_anio'].isin(valores_invalidos), 'estacion_anio'] = None

print(data['estacion_anio'].value_counts())
print(f"\nTotal categorías: {data['estacion_anio'].nunique()}")
print(f"Nulos en estacion_anio: {data['estacion_anio'].isna().sum()}")

Valores inválidos convertidos a NaN: 0
estacion_anio
primavera    624
invierno     532
otoño        529
verano       513
Name: count, dtype: int64

Total categorías: 4
Nulos en estacion_anio: 220


## **Validez**

1. **Fila corrupta 2015-07-13**: Afecta varias columnas a la vez con valores imposibles como (-9999, -1250, NaN), y no contamos con una regla que permita reconstruirla. Al no poder corregirla, se elimina para no inventar datos.

In [20]:
fila_corrupta = data[data['rafaga_min'] == -9999].copy()
print(f"Filas corruptas a eliminar: {fila_corrupta.shape[0]}")
data = data[data['rafaga_min'] != -9999]

Filas corruptas a eliminar: 1


2. **Presion_media:** Los valores >1050 son físicamente imposibles, pero al dividirlos entre 10 caen dentro del rango normal, puede ser un error de escritura y no creemos que sea un dato perdido. Usamoa .loc porque es una regla de negocio conocida, aplicable solo a los registros afectados.


In [21]:
mask_presion = (data['presion_media'] > 1050)
print(f"Registros corregidos en presion_media: {mask_presion.sum()}")
data.loc[mask_presion, 'presion_media'] = data.loc[mask_presion, 'presion_media'] / 10


Registros corregidos en presion_media: 5


3. **Humedad_media** Los valores <1 están en escala de proporción (0-1) en vez de porcentaje (0-100). Desidimos multiplicar por 100, pues recupera el dato real. Igual que con presión, es una corrección puntual con regla conocida, no una imputación.

In [22]:
mask_humedad = (data['humedad_media'] < 1)
print(f"Registros corregidos en humedad_media: {mask_humedad.sum()}")
data.loc[mask_humedad, 'humedad_media'] = data.loc[mask_humedad, 'humedad_media'] * 100

Registros corregidos en humedad_media: 775


4. **Registros_del_dia > 144:** Se encontraron 4 registros que superan el máximo lógico de 144 mediciones diarias. Al no poder determinarse la causa exacta del exceso de mediciones y no tratarse de columnas correlacionadas que permitan inferir el valor correcto, como sí pasaba con presión y humedad, se optó por eliminarlos en vez de intentar adivinar cuál sería el conteo correcto.

In [23]:
fuera_rango_registros = data[data['registros_del_dia'] > 144].copy()
print(f"Registros fuera de rango en registros_del_dia: {fuera_rango_registros.shape[0]}")
data = data[data['registros_del_dia'] <= 144]

Registros fuera de rango en registros_del_dia: 4


## **Completitud**

Todas las columnas tienen menos del 5% de nulos, dentro del rango donde el notebook recomienda imputación simple en lugar de eliminación masiva. Se usa mediana para las variables cuantitativas (ya sabemos que el dataset tuvo outliers, así que la mediana es más robusta que la media) y moda para las nominales, siguiendo la tabla de estrategias por tipo de variable. La columna fecha es la única excepción: al ser un identificador temporal sin valor numérico ni categórico que imputar, se eliminan sus filas nulas en vez de inventar una fecha.

In [24]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

temp_max_manana      0.036948
viento_min           0.034940
mes                  0.034538
estacion_anio        0.034538
anio                 0.034137
humedad_max          0.033333
viento_media         0.032530
presion_desv         0.031325
direccion_viento     0.031325
viento_norte         0.031325
rafaga_desv          0.030924
humedad_min          0.030924
viento_max           0.030924
rafaga_media         0.030522
humedad_media        0.029317
rafaga_min           0.028916
presion_media        0.028916
fecha                0.028916
rafaga_max           0.028514
presion_min          0.028112
sector_viento        0.028112
viento_desv          0.027309
dia_del_anio         0.026104
viento_este          0.025703
humedad_desv         0.024498
presion_max          0.024096
registros_del_dia    0.000000
dtype: float64

In [25]:
cuantitativas = [
    'presion_media', 'presion_min', 'presion_max', 'presion_desv',
    'humedad_media', 'humedad_min', 'humedad_max', 'humedad_desv',
    'viento_media', 'viento_min', 'viento_max', 'viento_desv',
    'rafaga_media', 'rafaga_min', 'rafaga_max', 'rafaga_desv',
    'viento_norte', 'viento_este', 'direccion_viento', 'temp_max_manana',
    'anio', 'dia_del_anio', 'registros_del_dia'
]

imputer_mediana = SimpleImputer(strategy='median')
data[cuantitativas] = imputer_mediana.fit_transform(data[cuantitativas])


In [26]:
cualitativas = ['estacion_anio', 'mes', 'sector_viento']

imputer_moda = SimpleImputer(strategy='most_frequent')
data[cualitativas] = imputer_moda.fit_transform(data[cualitativas])

In [27]:
filas_antes = data.shape[0]
data = data.dropna(subset=['fecha'])
print(f"Filas eliminadas por fecha nula: {filas_antes - data.shape[0]}")

Filas eliminadas por fecha nula: 72


In [28]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

fecha                0.0
rafaga_min           0.0
sector_viento        0.0
mes                  0.0
estacion_anio        0.0
dia_del_anio         0.0
anio                 0.0
registros_del_dia    0.0
direccion_viento     0.0
viento_este          0.0
viento_norte         0.0
rafaga_desv          0.0
rafaga_max           0.0
rafaga_media         0.0
presion_media        0.0
viento_desv          0.0
viento_max           0.0
viento_min           0.0
viento_media         0.0
humedad_desv         0.0
humedad_max          0.0
humedad_min          0.0
humedad_media        0.0
presion_desv         0.0
presion_max          0.0
presion_min          0.0
temp_max_manana      0.0
dtype: float64

## **Resumen de Limpieza:**


In [ ]:
print("=== Dimensiones del dataset ===")
print(f"Filas: {data.shape[0]}, Columnas: {data.shape[1]}")

print("\n=== Valores nulos ===")
print(data.isnull().sum()[data.isnull().sum() > 0])
if data.isnull().sum().sum() == 0:
    print("Sin valores nulos")

print("\n=== Duplicados ===")
dup = data.duplicated().sum()
print(f"Filas duplicadas: {dup}")
if dup == 0:
    print("Sin duplicados")

print("\n=== Valores únicos en sector_viento ===")
print(data['sector_viento'].value_counts())

print("\n=== Valores únicos en mes ===")
print(data['mes'].value_counts())

print("\n=== Rango de presion_media ===")
print(f"Min: {data['presion_media'].min()}, Max: {data['presion_media'].max()}")
if data['presion_media'].min() >= 950 and data['presion_media'].max() <= 1050:
    print("presion_media dentro del rango esperado")

print("\n=== Rango de humedad_media ===")
print(f"Min: {data['humedad_media'].min()}, Max: {data['humedad_media'].max()}")
if data['humedad_media'].min() >= 0 and data['humedad_media'].max() <= 100:
    print("humedad_media dentro del rango [0, 100]")

print("\n=== Rango de registros_del_dia ===")
print(f"Min: {data['registros_del_dia'].min()}, Max: {data['registros_del_dia'].max()}")
if data['registros_del_dia'].max() <= 144:
    print("registros_del_dia dentro del máximo lógico 144")

=== Dimensiones del dataset ===
Filas: 2418, Columnas: 27

=== Valores nulos ===
Series([], dtype: int64)
Sin valores nulos

=== Duplicados ===
Filas duplicadas: 0
Sin duplicados

=== Valores únicos en sector_viento ===
sector_viento
SO          800
S           541
NE          447
O           304
N           126
E            71
SE           65
NO           42
NORTH         6
EAST          4
Sureste       3
Oeste         3
Noroeste      3
East          3
Name: count, dtype: int64

=== Valores únicos en mes ===
mes
enero         275
julio         180
mayo          168
diciembre     162
octubre       154
agosto        154
marzo         153
septiembre    147
noviembre     144
febrero       136
junio         134
abril         117
NOVEMBER       29
APRIL          27
Nov            27
MARCH          27
Jun            27
Feb            27
SEPTEMBER      26
Aug            25
Apr            25
Sep            24
Sept           24
Dec            21
Mar            20
DECEMBER       19
Jul          

# **Ingenieria de Caracteristicas**

In [62]:
datatransf = data.copy()

## **Escalado de Variables Numericas**

Al igual que en el notebook, no todas las columnas se escalan. Para nuestro caso:
'presion_media', 'presion_min', 'presion_max', 'presion_desv', 'humedad_media', 'humedad_min', 'humedad_max', 'humedad_desv', 'viento_media', 'viento_min', 'viento_max', 'viento_desv', 'rafaga_media', 'rafaga_min', 'rafaga_max', 'rafaga_desv', 'viento_norte', 'viento_este', 'direccion_viento', 'registros_del_dia', 'dia_del_anio', puesto que son variables predictoras cuantitativas, con magnitudes muy distintas entre sí

In [40]:
columnas_a_escalar = [
    'presion_media', 'presion_min', 'presion_max', 'presion_desv',
    'humedad_media', 'humedad_min', 'humedad_max', 'humedad_desv',
    'viento_media', 'viento_min', 'viento_max', 'viento_desv',
    'rafaga_media', 'rafaga_min', 'rafaga_max', 'rafaga_desv',
    'viento_norte', 'viento_este', 'direccion_viento',
    'registros_del_dia', 'dia_del_anio'
]

scaler = MinMaxScaler()
datatransf[columnas_a_escalar] = scaler.fit_transform(datatransf[columnas_a_escalar])

print("Antes de escalar:")
print(data[['presion_media', 'viento_media', 'humedad_max']].head())
print("\nDespués de escalar:")
print(datatransf[['presion_media', 'viento_media', 'humedad_max']].head())

Antes de escalar:
   presion_media  viento_media  humedad_max
0       999.1456        0.7786         94.8
1       999.6006        1.4195         96.3
2       998.5486        1.2509         93.9
3       988.5107        1.7204         93.4
5       997.0530        1.2268         89.3

Después de escalar:
   presion_media  viento_media  humedad_max
0       0.800373      0.031454     0.928748
1       0.807634      0.057345     0.949301
2       0.790845      0.050534     0.916415
3       0.630643      0.069501     0.909564
5       0.766976      0.049560     0.853384


## **Codificacion de Variables Categoricas**

Aquí las tres variables categóricas son nominales con más de 2 categorías (sector_viento: 8, mes: 12, estacion_anio: 4), así que corresponde One-Hot Encoding, no codificación binaria:

In [66]:
datatransf = pd.get_dummies(datatransf, columns=['sector_viento', 'mes', 'estacion_anio'], dtype=int, drop_first=True)

print("Nuevas columnas generadas:")
print([col for col in datatransf.columns if col.startswith(('sector_viento_', 'mes_', 'estacion_anio_'))])

Nuevas columnas generadas:
['sector_viento_N', 'sector_viento_NE', 'sector_viento_NO', 'sector_viento_O', 'sector_viento_S', 'sector_viento_SE', 'sector_viento_SO', 'mes_agosto', 'mes_diciembre', 'mes_enero', 'mes_febrero', 'mes_julio', 'mes_junio', 'mes_marzo', 'mes_mayo', 'mes_noviembre', 'mes_octubre', 'mes_septiembre', 'estacion_anio_otoño', 'estacion_anio_primavera', 'estacion_anio_verano']


## **Creación de nuevas variables**

1. **Amplitud de presión, humedad y viento (amplitud_presion, amplitud_humedad, amplitud_viento):** estas variables capturan la variabilidad diaria de cada medición permitiendo identificar la diferencia entre el máximo y el mínimo del día, esta información no está explícita en las columnas originales. Un día con presión estable (amplitud baja) y un día con presión muy cambiante (amplitud alta) pueden tener la misma presion_media, pero representan condiciones climáticas muy distintas. La amplitud permite capturar esa diferencia, lo cual puede ser relevante para predecir temp_max_manana.

In [ ]:
datatransf['amplitud_presion'] = data['presion_max'] - data['presion_min']

datatransf['amplitud_humedad'] = data['humedad_max'] - data['humedad_min']

datatransf['amplitud_viento'] = data['viento_max'] - data['viento_min']

print("Estadísticas de las nuevas variables:")
print(datatransf[['amplitud_presion', 'amplitud_humedad', 'amplitud_viento']].describe().round(2))

Estadísticas de las nuevas variables:
       amplitud_presion  amplitud_humedad  amplitud_viento
count           2418.00           2418.00          2418.00
mean               5.02             45.18             6.19
std                5.16             32.58             7.87
min              -21.16            -68.75           -44.04
25%                2.83             25.10             3.02
50%                4.64             41.02             4.62
75%                7.22             62.18             8.40
max               65.51             99.46            44.06


2. **Velocidad resultante del viento(velocidad_resultante):** El dataset registra el viento descompuesto en dos componentes perpendiculares (viento_norte y viento_este), pero ninguna de las dos por separado representa la intensidad real del viento, un valor de viento_norte = 0 no significa que no hay viento, solo que el viento no sopla en esa dirección. Se calcula la magnitud del vector resultante (teorema de Pitágoras: √(norte² + este²)) para obtener una sola variable que sí refleja la fuerza total del viento, independientemente de su dirección.

In [72]:
datatransf['velocidad_resultante'] = (data['viento_norte']**2 + data['viento_este']**2) ** 0.5

print("Estadísticas de las nuevas variables:")
print(datatransf[['velocidad_resultante']].describe().round(2))

Estadísticas de las nuevas variables:
       velocidad_resultante
count               2418.00
mean                   1.72
std                    1.11
min                    0.00
25%                    0.85
50%                    1.50
75%                    2.36
max                    7.63


# **Resumen final del dataset transformado**

In [74]:
print("=== Dataset transformado (datatransf) ===")
print(f"Filas: {datatransf.shape[0]}, Columnas: {datatransf.shape[1]}")
print("\nTipos de datos:")
print(datatransf.dtypes)
print("\nPrimeras filas:")
datatransf.head()

=== Dataset transformado (datatransf) ===
Filas: 2418, Columnas: 49

Tipos de datos:
fecha                          str
presion_media              float64
presion_min                float64
presion_max                float64
presion_desv               float64
humedad_media              float64
humedad_min                float64
humedad_max                float64
humedad_desv               float64
viento_media               float64
viento_min                 float64
viento_max                 float64
viento_desv                float64
rafaga_media               float64
rafaga_min                 float64
rafaga_max                 float64
rafaga_desv                float64
viento_norte               float64
viento_este                float64
direccion_viento           float64
registros_del_dia          float64
anio                       float64
dia_del_anio               float64
temp_max_manana            float64
sector_viento_N              int64
sector_viento_NE             int64
secto

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,mes_noviembre,mes_octubre,mes_septiembre,estacion_anio_otoño,estacion_anio_primavera,estacion_anio_verano,velocidad_resultante,amplitud_presion,amplitud_humedad,amplitud_viento
0,2009-01-01,999.1456,996.500,1000.87,1.3993,91.0860,0.875000,94.8,1.7650,0.7786,...,0,0,0,0,0,0,0.362487,4.370,93.925000,1.509641
1,2009-01-02,999.6006,997.930,1002.65,1.5039,92.0868,86.600000,96.3,2.7588,1.4195,...,0,0,0,0,0,0,0.564057,4.720,9.700000,3.650000
2,2009-01-03,998.5486,993.050,1002.49,3.1304,76.4581,48.390000,93.9,15.1796,1.2509,...,0,0,0,0,0,0,0.875522,9.440,45.510000,3.520000
3,2009-01-04,988.5107,985.120,992.93,2.3223,89.4174,97.946704,93.4,4.4904,1.7204,...,0,0,0,0,0,0,1.534270,7.810,-4.546704,1.914415
5,2009-01-06,997.0530,987.075,998.49,1.2689,83.7747,67.310000,89.3,4.8864,1.2268,...,0,0,0,0,0,0,0.681033,11.415,21.990000,1.925089


# **Conclusion**

En esta etapa se transformó el dataset meteorológico crudo (2576 filas, 27 columnas) en un conjunto de datos limpio y listo para modelar, siguiendo las cuatro dimensiones de calidad trabajadas en el laboratorio anterior:

Validez: Se corrigieron valores fuera del dominio esperado de cada variable, incluyendo un error de digitación en presion_media (6 registros que se debían dividir entre 10 por estar fuera de rango), una mezcla de escalas en humedad_media (776 registros registrados como proporción 0 a 1 en vez de porcentaje), un registro con múltiples valores imposibles simultáneos (rafaga_min = -9999, viento_norte = -1250) que se eliminó por no poder reconstruirse, y 4 registros de registros_del_dia que superaban el máximo lógico de 144 mediciones diarias.

Unicidad: Se eliminaron 4 filas completamente duplicadas mediante drop_duplicates(), confirmadas tanto manualmente como por el reporte automático de ydata profiling.

Consistencia: Se normalizaron las tres variables categóricas (sector_viento, mes, estacion_anio), que originalmente tenían entre 12 y 28 variantes de una misma categoría por mezcla de mayúsculas, idioma (español e inglés) y formato (abreviado o completo). En estacion_anio además se identificaron valores fuera de dominio (Estacion_Desconocida, East, VERANO_INVIERNO) que se convirtieron a nulos en lugar de forzarlos a una categoría incorrecta.

Completitud: Todas las 27 columnas presentaban valores nulos (entre 2.4% y 3.7%, más los nulos nuevos generados al invalidar categorías erróneas). Se imputaron con mediana las variables cuantitativas, por su robustez ante los outliers ya identificados, y con moda las variables categóricas nominales.

Una vez corregidos estos problemas, se aplicó ingeniería de características sobre una copia independiente del dataset limpio, escalado con MinMaxScaler a las variables predictoras cuantitativas, evitando que su magnitud domine artificialmente los cálculos de distancia de algoritmos como KNN o regresión, codificación con One Hot Encoding (drop_first=True) para las tres variables categóricas nominales, y la creación de cuatro nuevas variables (amplitud_presion, amplitud_humedad, amplitud_viento, velocidad_resultante) que capturan relaciones de negocio no explícitas en los datos originales.